<a href="https://colab.research.google.com/github/yashaskarsrivastava-source/CodeAlpha_CreditScoringModel/blob/main/ai_ml_hack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import zipfile
import os

# Apni zip file ka naam yahan daalna (jo bhi Colab mein upload ki thi)
zip_path = "/content/c215051c-6-Archive 4 (1) (1).zip"  # <-- apna exact filename check kar lena

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/dataset")

# Confirm karne ke liye ki files kaha extract hui hain
for root, dirs, files in os.walk("/content/dataset"):
    for f in files:
        print(os.path.join(root, f))

/content/dataset/sample_submission.csv
/content/dataset/train.csv
/content/dataset/test.csv
/content/dataset/__MACOSX/._test.csv
/content/dataset/__MACOSX/._train.csv
/content/dataset/__MACOSX/._sample_submission.csv


In [3]:
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import NearestNeighbors

# ---------------------------------------------------------
# 1. Load data (apna path check kar lena)
# ---------------------------------------------------------
train = pd.read_csv("/content/dataset/train.csv")
test = pd.read_csv("/content/dataset/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# ---------------------------------------------------------
# 2. Clean text
# ---------------------------------------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["clean_review"] = train["Reviews"].apply(clean_text)
test["clean_review"] = test["Reviews"].apply(clean_text)

# ---------------------------------------------------------
# 3. Raw term counts (fit vocab on train only)
# ---------------------------------------------------------
cv = CountVectorizer(
    max_features=100000,
    ngram_range=(1, 3),
    min_df=1,
    max_df=0.3,
    stop_words="english",
)
X_train_tf = cv.fit_transform(train["clean_review"])
X_test_tf = cv.transform(test["clean_review"])

N = X_train_tf.shape[0]

# ---------------------------------------------------------
# 4. BM25 weighting
# ---------------------------------------------------------
df = np.asarray((X_train_tf > 0).sum(axis=0)).ravel()
idf = np.log((N - df + 0.5) / (df + 0.5) + 1.0)

doc_len_train = np.asarray(X_train_tf.sum(axis=1)).ravel()
avgdl = doc_len_train.mean()
doc_len_test = np.asarray(X_test_tf.sum(axis=1)).ravel()

k1, b = 1.2, 0.25

def bm25_transform(X_tf, doc_len, idf, k1, b, avgdl):
    X = X_tf.tocsr().astype(np.float64).copy()
    row_norm = k1 * (1 - b + b * doc_len / avgdl)
    nnz_per_row = np.diff(X.indptr)
    row_norm_per_nnz = np.repeat(row_norm, nnz_per_row)
    tf = X.data
    X.data = tf * (k1 + 1) / (tf + row_norm_per_nnz)
    X = X.multiply(idf).tocsr()
    return X.astype(np.float32)

X_train_bm25 = bm25_transform(X_train_tf, doc_len_train, idf, k1, b, avgdl)
X_test_bm25 = bm25_transform(X_test_tf, doc_len_test, idf, k1, b, avgdl)

# ---------------------------------------------------------
# 5. Nearest Neighbors (cosine similarity on BM25 vectors)
# ---------------------------------------------------------
nn = NearestNeighbors(n_neighbors=10, metric="cosine", algorithm="brute", n_jobs=-1)
nn.fit(X_train_bm25)

batch_size = 500
all_neighbor_positions = []
for start in range(0, X_test_bm25.shape[0], batch_size):
    end = min(start + batch_size, X_test_bm25.shape[0])
    _, neigh_idx = nn.kneighbors(X_test_bm25[start:end])
    all_neighbor_positions.append(neigh_idx)
    print(f"Processed {end}/{X_test_bm25.shape[0]}")

neighbor_positions = np.vstack(all_neighbor_positions)

# ---------------------------------------------------------
# 6. Map positions -> actual train.csv Index values
# ---------------------------------------------------------
train_index_array = train["Index"].values
neighbor_real_indices = train_index_array[neighbor_positions]

# ---------------------------------------------------------
# 7. Format as bracketed string like sample_submission.csv
# ---------------------------------------------------------
def format_bracket_list(idx_list):
    return "[" + ", ".join(str(int(x)) for x in idx_list) + "]"

test["Index_list"] = [format_bracket_list(row) for row in neighbor_real_indices]

# ---------------------------------------------------------
# 8. Save submission
# ---------------------------------------------------------
submission = test[["Index", "Index_list"]]
assert submission.shape == (10977, 2), f"Shape mismatch: {submission.shape}"

submission.to_csv("submission.csv", index=False)
print("submission.csv saved. Shape:", submission.shape)
print(submission.head())

from google.colab import files
files.download('submission.csv')

Train shape: (109776, 3)
Test shape: (10977, 2)
Processed 500/10977
Processed 1000/10977
Processed 1500/10977
Processed 2000/10977
Processed 2500/10977
Processed 3000/10977
Processed 3500/10977
Processed 4000/10977
Processed 4500/10977
Processed 5000/10977
Processed 5500/10977
Processed 6000/10977
Processed 6500/10977
Processed 7000/10977
Processed 7500/10977
Processed 8000/10977
Processed 8500/10977
Processed 9000/10977
Processed 9500/10977
Processed 10000/10977
Processed 10500/10977
Processed 10977/10977
submission.csv saved. Shape: (10977, 2)
    Index                                         Index_list
0  109776  [47258, 88353, 5332, 29734, 54730, 94371, 8266...
1  109777  [105353, 67136, 106992, 28204, 98313, 53862, 3...
2  109778  [31137, 103177, 73695, 1455, 98603, 96854, 380...
3  109779  [37568, 48556, 35290, 4566, 33630, 34652, 5986...
4  109780  [96443, 84688, 49239, 106819, 9305, 98597, 851...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>